# DeepSeek-OCR on Google Colab

1. Select **Runtime → Change runtime type → GPU**.
2. Run the setup cells from top to bottom through **Choose the GPU dtype**. The first OCR run downloads the model weights (about 6.7 GB).
3. The notebook clones and checks [this public repository](https://github.com/ubaid-148/deeksheekocr). The smoke test is optional; the last cell extracts structured invoice fields from an image or PDF and downloads one JSON file.

Inference follows the [upstream vLLM DeepSeek-OCR recipe](https://docs.vllm.ai/projects/recipes/en/latest/DeepSeek/DeepSeek-OCR.html). Colab GPU availability varies; if no GPU is assigned, reconnect with a GPU runtime.

In [ ]:
import shutil
import subprocess

if not shutil.which('nvidia-smi'):
    raise RuntimeError('Select a GPU runtime: Runtime → Change runtime type → GPU')
subprocess.run(['nvidia-smi'], check=True)

## Clone and check the public repository

The repository has Python scripts rather than a compiled build target. This cell checks every Python source file without creating build files.

In [ ]:
import ast
from pathlib import Path

REPO_URL = 'https://github.com/ubaid-148/deeksheekocr.git'
REPO_DIR = Path('/content/deeksheekocr')
if REPO_DIR.exists():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)], check=True)

sources = sorted(REPO_DIR.rglob('*.py'))
if not sources:
    raise RuntimeError('No Python files found in the public repository')
for source in sources:
    ast.parse(source.read_text(encoding='utf-8'), filename=str(source))
print(f'Checked {len(sources)} Python files in {REPO_DIR}')
subprocess.run(['git', '-C', str(REPO_DIR), 'status', '--short', '--branch'], check=True)

## Install the GPU runtime

vLLM and its PyTorch packages are installed in an isolated environment. This keeps Colab's preinstalled TorchAudio and Pillow from mixing with vLLM's dependencies. The repository's older Transformers `requirements.txt` is not used here.

In [ ]:
import sys

VENV_DIR = Path('/content/deepseek-ocr-venv')
VENV_PYTHON = VENV_DIR / 'bin' / 'python'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'uv'], check=True)
if not VENV_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'uv', 'venv', str(VENV_DIR), '--python', sys.executable], check=True)
subprocess.run([
    sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(VENV_PYTHON),
    'vllm', 'pillow', 'soxr', 'pymupdf', '--torch-backend=auto',
], check=True)

import_check = [str(VENV_PYTHON), '-c', 'from PIL import Image, ImageText; import pymupdf; from vllm import LLM, SamplingParams; from vllm.sampling_params import StructuredOutputsParams; from vllm.model_executor.models.deepseek_ocr import NGramPerReqLogitsProcessor']
checked = subprocess.run(import_check, capture_output=True, text=True)
if checked.returncode != 0 and "cannot import name '_Ink'" in checked.stderr:
    print('Repairing Pillow in the isolated environment')
    subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--python', str(VENV_PYTHON), '--reinstall-package', 'pillow', 'pillow'], check=True)
    checked = subprocess.run(import_check, capture_output=True, text=True)
if checked.returncode != 0:
    raise RuntimeError('Isolated-environment import failed:\n' + checked.stderr)
print('vLLM, Pillow, and PyMuPDF import correctly from', VENV_DIR)

## Choose the GPU dtype

T4 GPUs need float16. A100 and newer GPUs use bfloat16. OCR and invoice extraction run as separate processes using the isolated environment's Python.

In [ ]:
probe = subprocess.check_output([
    str(VENV_PYTHON), '-c',
    'import torch; assert torch.cuda.is_available(); print(*torch.cuda.get_device_capability(0), sep=".")'
], text=True).strip()
gpu_major, gpu_minor = map(int, probe.splitlines()[-1].split('.'))
if (gpu_major, gpu_minor) < (7, 5):
    raise RuntimeError('vLLM needs an NVIDIA GPU with compute capability 7.5 or newer')
model_dtype = 'float16' if gpu_major < 8 else 'bfloat16'
print('GPU compute capability:', f'{gpu_major}.{gpu_minor}', 'model dtype:', model_dtype)
runner = REPO_DIR / 'colab_infer.py'
extractor = REPO_DIR / 'colab_extract_invoice.py'
for script in (runner, extractor):
    if not script.is_file():
        raise FileNotFoundError(f'Colab script missing from repository: {script}')

## Smoke test (optional)

In [ ]:
smoke_output = Path('/content/deepseek_ocr_smoke.md')
subprocess.run([
    str(VENV_PYTHON), str(runner), '--smoke', '--dtype', model_dtype,
    '--output', str(smoke_output),
], check=True)
print('Smoke test saved to', smoke_output)

## Extract an invoice as structured JSON

Run this cell after setup. Upload one invoice as PNG, JPG, JPEG, or PDF. DeepSeek-OCR reads every page, then [Qwen2.5-1.5B-Instruct](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct) fills a fixed invoice JSON schema. The second model downloads once on first use (about 3 GB). Unknown fields are `null`; the result includes a `review_required` flag. The final JSON appears below this cell and downloads as `<filename>_invoice.json`; only that structured file is downloaded.

In [ ]:
from google.colab import files

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload exactly one PNG, JPG, JPEG, or PDF file')
source_name = next(iter(uploaded))
del uploaded
suffix = Path(source_name).suffix.lower()
if suffix not in {'.png', '.jpg', '.jpeg', '.pdf'}:
    raise ValueError('Only PNG, JPG, JPEG, and PDF are supported in this cell')
source_flag = '--pdf' if suffix == '.pdf' else '--image'
work_dir = Path('/content/deepseek_ocr_work')
work_dir.mkdir(parents=True, exist_ok=True)
ocr_text = work_dir / f'{Path(source_name).stem}_ocr.md'
ocr_json = work_dir / f'{Path(source_name).stem}_ocr.json'
invoice_json = Path('/content') / f'{Path(source_name).stem}_invoice.json'
ocr_command = [
    str(VENV_PYTHON), str(runner), source_flag, source_name, '--dtype', model_dtype,
    '--output', str(ocr_text), '--json-output', str(ocr_json),
]
extract_command = [
    str(VENV_PYTHON), str(extractor), '--ocr-json', str(ocr_json),
    '--output', str(invoice_json), '--dtype', model_dtype,
]
def run_live(command):
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        return_code = process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, command)
print('Step 1/2: Reading invoice text...', flush=True)
run_live(ocr_command)
print('Step 2/2: Building structured invoice JSON...', flush=True)
run_live(extract_command)
files.download(str(invoice_json))